In [4]:
from datetime import datetime, timedelta

In [ ]:
class Member:
    def __init__(self, id_member, name, role):
        self.id_member = id_member
        self.name = name
        self.role = role

class Resource:
    def __init__(self, id_resource, name):
        self.id_resource = id_resource
        self.name = name

class Workstation(Resource):
    def __init__(self, id_resource, name, gpu):
        super().__init__(id_resource, name)
        self.gpu = gpu

    def check_capacity(self, user):
        return True

class Meetingroom(Resource):
    def __init__(self, id_resource, name, maxcapacity):
        super().__init__(id_resource, name)
        self.maxcapacity = maxcapacity

    def check_capacity(self, user):
        if not user > self.maxcapacity:
            return True
        return False

class Reservation:
    def __init__(self, id_reservation, member, resource, starttime, duration, status):
        self.id_reservation = id_reservation
        self.member = member
        self.resource = resource
        self.starttime = starttime
        self.duration = duration
        #selisih
        difference = timedelta(hours=duration)
        self.finishtime = starttime + difference

        self.status = 'ACTIVE'

    def checkfull(self, new_starttime, new_finishtime):
        if self.status == 'CANCELLED':
            return False

        if new_starttime < self.finishtime and new_finishtime > self.starttime:
            return True
        return False

class LabSystem:
    def __init__(self):
        self.member = []
        self.resource = []
        self.reservation = []
        self.no_queue = 1

    def add_member(self, member):
        self.member.append(member)

    def add_resource(self, resource):
        self.resource.append(resource)

    def make_reservation(self, id_member, id_resource, starttime, duration, user=1):
        found_member = None
        for m in self.member:
            if m.id_member == id_member:
                found_member = m
                break
        if not found_member:
            print("Gagal: Member tidak terdaftar.")
            return None

        # 2. Cari Resource
        found_resource = None
        for r in self.resource:
            if r.id_resource == id_resource:
                found_resource = r
                break
        if not found_resource:
            print("Gagal: Resource tidak ditemukan.")
            return None

        # 3. Validasi Durasi dan Kapasitas
        if duration <= 0:
            print("Gagal: Durasi harus lebih dari 0 jam.")
            return None

        if not found_resource.check_capacity(user):
            print(f"Gagal: Kapasitas {found_resource.name} tidak cukup untuk {user} orang.")
            return None

        # 4. Cek Bentrok Jadwal
        new_finishtime = starttime + timedelta(hours=duration)
        for res in self.reservation:
            if res.resource.id_resource == id_resource and res.checkfull(starttime, new_finishtime):
                print(f"Gagal: Jadwal {found_resource.name} bentrok pada jam tersebut.")
                return None

        # 5. Buat dan Simpan Reservasi
        res_id = f"RES-{self.no_queue}"
        self.no_queue += 1

        new_res = Reservation(res_id, found_member, found_resource, starttime, duration)
        self.reservation.append(new_res)
        print(f"Sukses: {found_member.name} berhasil memesan {found_resource.name} ({res_id})")
        return new_res

    def cancel_reservation(self, id_reservation):
        for res in self.reservation:
            if res.id_reservation == self.id_reservation:
                if res.status == 'CANCELLED':
                    print("Gagal: sudah pernah di batalkan sebelumnya..")
                    return
                res.status == 'CANCELLED'
                print(f"Sukses: Reservasi {id_reservation} berhasil di cancel")
                return
            print("Gagal: ID Reservasi tidak ditemukan")
        
print("works")


IndentationError: expected an indented block after function definition on line 17 (1501121512.py, line 19)

In [7]:
lab = LabSystem()

# Input Member
lab.add_member(Member("M001", "Andi", "Student"))
lab.add_member(Member("M002", "Sarah", "Assistant"))
lab.add_member(Member("M003", "Budi", "Student"))

# Input Resource
lab.add_resource(Workstation("WS01", "AI Workstation 1", "RTX 4090"))
lab.add_resource(Meetingroom("MR01", "Discussion Room", 8))

print("--- MENJALANKAN SKENARIO WAJIB ---")

# Skenario A: Andi pesan AI Workstation 1 jam 09:00 selama 2 jam (Berhasil)
res_A = lab.make_reservation("M001", "WS01", datetime(2026, 9, 10, 9, 0), 2)

# Skenario B: Sarah coba pesan AI Workstation 1 jam 10:00 selama 2 jam (Ditolak overlap)
res_B = lab.make_reservation("M002", "WS01", datetime(2026, 9, 10, 10, 0), 2)

# Skenario C: Sarah pesan AI Workstation 1 jam 11:00 selama 1 jam (Berhasil)
res_C = lab.make_reservation("M002", "WS01", datetime(2026, 9, 10, 11, 0), 1)

# Skenario D: Budi pesan Discussion Room jam 13:00 untuk 10 orang (Ditolak, kapasitas 8)
res_D = lab.make_reservation("M003", "MR01", datetime(2026, 9, 11, 13, 0), 2, user=10)

# Skenario E: Andi membatalkan reservasi pertamanya
if res_A:
    lab.cancel_reservation(res_A.id_reservation)

--- MENJALANKAN SKENARIO WAJIB ---


AttributeError: 'Workstation' object has no attribute 'check_capacity'